In [1]:
# ==============================================================================
# CELL 1: ENVIRONMENT VERIFICATION & DEPENDENCY SETUP (ROBUST VERSION)
# ==============================================================================
import os
import json
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)

# 1. Hardware acceleration verification
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[SYSTEM AUDIT] Execution Device: {device}")
if device == "cuda":
    print(f"[SYSTEM AUDIT] Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"[SYSTEM AUDIT] Device Count: {torch.cuda.device_count()}")
else:
    print("[WARNING] Running on CPU! Execution will be significantly throttled.")

# 2. Dynamic Asset Discovery
def find_file(filename, search_root="/kaggle/input"):
    for root, _, files in os.walk(search_root):
        if filename in files:
            return os.path.join(root, filename)
    # Check local fallback
    if os.path.exists(filename):
        return filename
    return None

CORPUS_PATH = find_file("bilingual_med_legal_corpus.txt")
TOKENIZER_JSON_PATH = find_file("med_legal_tokenizer.json")

print(f"[SYSTEM AUDIT] Corpus Path: {CORPUS_PATH}")
print(f"[SYSTEM AUDIT] Tokenizer Path: {TOKENIZER_JSON_PATH}")

assert CORPUS_PATH is not None, "Error: Corpus file not found in input datasets!"
assert TOKENIZER_JSON_PATH is not None, "Error: Tokenizer JSON file not found in input datasets!"
print("[SYSTEM AUDIT] Artifacts located successfully.")


[SYSTEM AUDIT] Execution Device: cpu
[WARNING] Running on CPU! Execution will be significantly throttled.
[SYSTEM AUDIT] Corpus Path: bilingual_med_legal_corpus.txt
[SYSTEM AUDIT] Tokenizer Path: med_legal_tokenizer.json
[SYSTEM AUDIT] Artifacts located successfully.


In [4]:
# ==============================================================================
# CELL 2: VOCABULARY EXTRACTION & MINING
# ==============================================================================
print(">>> Stage 1: Mining High-Density Domain Tokens...")

base_model_name = "xlm-roberta-base"
base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)

# Ingest custom-trained domain tokenizer vocabulary
with open(TOKENIZER_JSON_PATH, "r", encoding="utf-8") as f:
    custom_vocab = json.load(f)["model"]["vocab"]

base_vocab_set = set(base_tokenizer.get_vocab().keys())
print(base_vocab_set)

# Filter tokens: absent from base model & minimum length threshold
tokens_to_add = []
for raw_token in custom_vocab.keys():
    clean_token = raw_token.replace("Ġ", "").strip()
    if clean_token and len(clean_token) >= 3 and clean_token not in base_vocab_set:
        tokens_to_add.append(clean_token)

selected_new_tokens = list(set(tokens_to_add))[:2000]
print(selected_new_tokens)
print(f"[OK] Filtered {len(selected_new_tokens)} distinct domain tokens for injection.")

>>> Stage 1: Mining High-Density Domain Tokens...
{'▁práca', '▁yor', '▁ಕೆಲವು', '▁했다', 'bin', '▁దరఖాస్తు', 'ဖြစ်စေ', '▁termékek', '出來的', 'cja', '▁Él', 'ចែករំលែក', 'γγ', '▁fonda', 'ຮູ້', '▁მოგე', '▁sonder', 'سکو', 'ံုး', 'ສະຫວັນ', '▁defect', '▁2015-2016', '▁mata', '▁оз', '蛛', 'крит', 'UNDE', 'TIF', '▁JUN', '▁pipi', '▁Jelly', '▁Prezes', '▁шү', '检验', '▁egokia', 'คณะกรรมการ', '▁ભોગ', '▁Disciplin', '加工', '▁հասց', '荣誉', '福音', '▁жаахан', 'ပို့', '▁ваздушн', '▁سردار', '▁проверен', 'သတ္မွတ္', '还真', 'mish', 'ԱՆ', '▁TR', '୍ୟୁ', '▁ਪੁਸਤਕ', 'жете', '▁சேர்க்க', '▁그렇지', '▁එකෙන්', '🧚', 'родження', '▁défense', 'हल', '▁Trời', '▁pluri', 'byt', '▁ligj', 'ttelee', '▁nõu', '岵', 'นครสวรรค์', '▁ນັ້ນ', 'missa', '▁ಇರಲಿಲ್ಲ', '▁आहात', 'mati', '渊', 'ητικό', 'эты', '▁wystaw', '▁විශ්වවිද්', '▁tím', 'ტი', 'ไหน', '▁memeriksa', '▁Mě', '▁požiar', '689', '▁कामदार', '▁հարցին', '▁mjere', '▁sona', 'dım', '▁پڑھنے', '▁글을', 'ET', '▁votar', '▁عشر', 'УМ', '▁заместник', '▁детстве', '▁maravilla', '景色', '▁의료', '▁اسلامی', '靠近', '▁tröt

In [3]:
# ==============================================================================
# CELL 3: TENSOR SURGERY & FREEZING
# ==============================================================================
print(">>> Stage 2: Performing Tensor Surgery...")

model = AutoModelForMaskedLM.from_pretrained(base_model_name)

# Expand tokenizer & embedding matrix dimensions
num_added = base_tokenizer.add_tokens(selected_new_tokens)
model.resize_token_embeddings(len(base_tokenizer))

# Freeze transformer backbone entirely (prevent weight corruption)
for param in model.roberta.parameters():
    param.requires_grad = False

# Unfreeze strictly input embeddings for the warm-up phase
input_embeddings = model.get_input_embeddings()
input_embeddings.weight.requires_grad = True

# Audit trainable capacity
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"[AUDIT] Trainable Parameters: {trainable_params:,} / {total_params:,} "
      f"({100 * trainable_params / total_params:.2f}%)")

>>> Stage 2: Performing Tensor Surgery...


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


[AUDIT] Trainable Parameters: 194,381,666 / 279,833,186 (69.46%)


In [4]:
# ==============================================================================
# CELL 4: DATA COLLATOR & STREAM PIPELINE
# ==============================================================================
print(">>> Stage 3: Building Masked Data Pipeline...")

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    corpus_lines = [line.strip() for line in f if len(line.strip()) > 30][:25000]

raw_dataset = Dataset.from_dict({"text": corpus_lines})

def tokenize_batch(batch):
    return base_tokenizer(
        batch["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )

# Parallel tokenization across available cores
tokenized_dataset = raw_dataset.map(
    tokenize_batch,
    batched=True,
    num_proc=2,
    remove_columns=["text"],
    desc="Tokenizing dataset on cloud"
)

# Dynamic Masked Language Modeling collator (15% random masking)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=base_tokenizer,
    mlm=True,
    mlm_probability=0.15
)
print("[OK] Dataset transformed and masking collator ready.")

>>> Stage 3: Building Masked Data Pipeline...


Tokenizing dataset on cloud (num_proc=2):   0%|          | 0/25000 [00:00<?, ? examples/s]

[OK] Dataset transformed and masking collator ready.


In [5]:
# ==============================================================================
# CELL 5: CLOUD WARM-UP OPTIMIZATION (OOM-PROTECTED ARCHITECTURE)
# ==============================================================================
import os
import torch
from transformers import Trainer, TrainingArguments

print(">>> Stage 4: Launching OOM-Protected GPU Training Engine...")

OUTPUT_DIR = "/kaggle/working/med_legal_adapted_checkpoint"
EXPORT_DIR = "/kaggle/working/med_legal_base_model"

# Enable dynamic memory allocation to prevent fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    
    # 1. Memory Throttle: Slash physical footprint per step
    per_device_train_batch_size=16,     # Scaled down from 64 to fit the 252k-vocab tensor
    gradient_accumulation_steps=4,      # 16 x 4 = Effective batch size of 64 preserved
    
    # 2. Optimization Parameters
    learning_rate=1e-3,                 # Aggressive step size for embedding warm-up
    weight_decay=0.01,
    logging_steps=10,
    save_steps=200,
    save_total_limit=1,
    
    # 3. Hardware Acceleration
    fp16=torch.cuda.is_available(),     # Half-precision FP16 activation
    dataloader_num_workers=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# Launch optimization
trainer.train()

# Export finalized weights and adapted tokenizer to Kaggle Output
model.save_pretrained(EXPORT_DIR)
base_tokenizer.save_pretrained(EXPORT_DIR)

print(f"\n[SUCCESS] Model weights fully synchronized and exported to: {EXPORT_DIR}")

>>> Stage 4: Launching OOM-Protected GPU Training Engine...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
10,25.034222
20,23.372585
30,22.840463
40,20.792892
50,20.244508
60,19.312695
70,18.347935
80,18.046858
90,17.538963
100,16.969669


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


[SUCCESS] Model weights fully synchronized and exported to: /kaggle/working/med_legal_base_model
